<a href="https://colab.research.google.com/github/Boopalan-d3v/collab-tests/blob/main/clip_zero_spot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Cell 1: Install required libraries
# ============================================================
!pip install -q pymupdf transformers torch torchvision pillow pandas tqdm

In [ ]:
# ============================================================
# Cell 2: Import libraries
# ============================================================
import os
import shutil
from pathlib import Path

import fitz
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

In [ ]:
# ============================================================
# Cell 3: Configuration
# ============================================================
PDF_PATH = "/kaggle/input/datasets/boopaland3v/multiple-flowchart/The_European_Medical_Device_Regulation-SECURED.pdf"

WORK_DIR = Path("/kaggle/working/clip_flowchart_extraction")
IMAGE_DIR = WORK_DIR / "embedded_images"
CANDIDATE_DIR = WORK_DIR / "flowchart_candidates"
REJECT_DIR = WORK_DIR / "rejected_images"

for folder in [IMAGE_DIR, CANDIDATE_DIR, REJECT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("PDF path:", PDF_PATH)
print("Output folder:", WORK_DIR)

In [ ]:
# ============================================================
# Cell 4: Extract embedded images from PDF using PyMuPDF
# ============================================================
doc = fitz.open(PDF_PATH)
image_records = []

for page_index in tqdm(range(len(doc)), desc="Extracting images"):
    page = doc[page_index]
    images = page.get_images(full=True)

    for image_index, img in enumerate(images):
        xref = img[0]

        try:
            extracted = doc.extract_image(xref)
            image_bytes = extracted["image"]
            image_ext = extracted["ext"]

            image_name = f"page_{page_index + 1}_img_{image_index + 1}.{image_ext}"
            image_path = IMAGE_DIR / image_name

            with open(image_path, "wb") as f:
                f.write(image_bytes)

            image_records.append({
                "page_number": page_index + 1,
                "image_index": image_index + 1,
                "image_path": str(image_path),
                "extension": image_ext
            })

        except Exception as e:
            print(f"Failed on page {page_index + 1}, image {image_index + 1}: {e}")

df_images = pd.DataFrame(image_records)

print("Total embedded images extracted:", len(df_images))
df_images.head()

In [ ]:
# ============================================================
# Cell 5: Remove small images such as logos/icons
# ============================================================
MIN_WIDTH = 250
MIN_HEIGHT = 180
MIN_AREA = 60000

size_records = []

for _, row in df_images.iterrows():
    try:
        img = Image.open(row["image_path"])
        width, height = img.size
        area = width * height

        size_records.append({
            **row.to_dict(),
            "width": width,
            "height": height,
            "area": area
        })

    except Exception as e:
        print("Image read error:", row["image_path"], e)

df_sized = pd.DataFrame(size_records)

df_big = df_sized[
    (df_sized["width"] >= MIN_WIDTH) &
    (df_sized["height"] >= MIN_HEIGHT) &
    (df_sized["area"] >= MIN_AREA)
].copy()

print("Images before size filtering:", len(df_sized))
print("Images after size filtering:", len(df_big))

df_big.head()

In [ ]:
# ============================================================
# Cell 6: Load CLIP zero-shot model
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

MODEL_NAME = "openai/clip-vit-base-patch32"

clip_model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
clip_processor = CLIPProcessor.from_pretrained(MODEL_NAME)

print("CLIP model loaded successfully")

In [ ]:
# ============================================================
# Cell 7: Define zero-shot labels
# ============================================================
LABELS = [
    "a flowchart diagram with decision boxes arrows and process steps",
    "a data table with rows and columns",
    "a hierarchy tree diagram",
    "a simple document illustration",
    "a logo or small icon",
    "a normal photograph"
]

In [ ]:
# ============================================================
# Cell 8: CLIP classification function
# ============================================================
def classify_image_with_clip(image_path):
    """
    Classifies one image using CLIP zero-shot classification.
    Returns best label, best score, and all label scores.
    """

    image = Image.open(image_path).convert("RGB")

    inputs = clip_processor(
        text=LABELS,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = clip_model(**inputs)
        probs = outputs.logits_per_image.softmax(dim=1)[0]

    scores = {
        LABELS[i]: float(probs[i])
        for i in range(len(LABELS))
    }

    best_label = max(scores, key=scores.get)
    best_score = scores[best_label]

    return best_label, best_score, scores

In [ ]:
# ============================================================
# Cell 9: Run CLIP classification on extracted images
# ============================================================
clip_records = []

for _, row in tqdm(df_big.iterrows(), total=len(df_big), desc="Classifying images"):
    best_label, best_score, scores = classify_image_with_clip(row["image_path"])

    clip_records.append({
        **row.to_dict(),
        "best_label": best_label,
        "best_score": best_score,
        "flowchart_score": scores["a flowchart diagram with decision boxes arrows and process steps"],
        "table_score": scores["a data table with rows and columns"],
        "tree_score": scores["a hierarchy tree diagram"],
        "document_score": scores["a simple document illustration"],
        "logo_score": scores["a logo or small icon"],
        "photo_score": scores["a normal photograph"],
    })

df_clip = pd.DataFrame(clip_records)

df_clip[
    ["page_number", "image_index", "best_label", "best_score",
     "flowchart_score", "table_score", "tree_score"]
]

In [ ]:
# ============================================================
# Cell 10: Decide final flowchart candidates
# ============================================================
def is_flowchart_candidate(row):
    """
    Keeps images where flowchart score is stronger than table/tree/document scores.
    Adjust threshold if needed.
    """

    return (
        row["flowchart_score"] >= 0.30 and
        row["flowchart_score"] > row["table_score"] and
        row["flowchart_score"] > row["tree_score"]
    )

df_clip["classification"] = df_clip.apply(
    lambda row: "flowchart_candidate" if is_flowchart_candidate(row) else "rejected",
    axis=1
)

df_clip["classification"].value_counts()

In [ ]:
# ============================================================
# Cell 11: Copy candidates and rejected images into folders
# ============================================================
for _, row in df_clip.iterrows():
    src = Path(row["image_path"])

    if row["classification"] == "flowchart_candidate":
        dst = CANDIDATE_DIR / src.name
    else:
        dst = REJECT_DIR / src.name

    shutil.copy(src, dst)

print("Flowchart candidates:", len(list(CANDIDATE_DIR.glob("*"))))
print("Rejected images:", len(list(REJECT_DIR.glob("*"))))

In [ ]:
# ============================================================
# Cell 12: Display detected flowchart candidates
# ============================================================
from IPython.display import display

candidate_files = list(CANDIDATE_DIR.glob("*"))

for img_path in candidate_files[:30]:
    print(img_path.name)
    display(Image.open(img_path))